# 06 — Artifact registry, validation, and manuscript sync

**Purpose.** Register every retained paper artifact (generated inside the thematic
notebooks 01–05), validate the full contract — headline values, catalog attention-change
values, visibility-support reconciliation, typographic and content-integrity
checks — refresh the manuscript's `Figures/` and `Tables/` copies from `outputs/`,
and verify that every LaTeX reference resolves.

**Inputs.** The result objects in `outputs/data/results/` and the artifacts in
`outputs/tables/` and `outputs/figures/` written by notebooks 01–05.

**Outputs.** `outputs/data/artifact_trace.csv`, `outputs/data/validation_results.csv`,
and the synchronized `manuscript/Figures/` and `manuscript/Tables/` copies.
Manual validation additionally reads the preserved review CSV and
`data/validation/camila_linkage_error_annotations.csv`, and writes enriched
review records and summaries under `outputs/data/validation/`.


In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw_checksums.csv").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from IPython.display import display

from cdr import paths, build, util
from cdr.util import SNAPSHOTS, SNAPSHOT_LABEL, check, show_and_save_table, show_and_save_figure

paths.ensure_output_dirs()
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
RESULTS = paths.RESULTS

In [2]:
from cdr import tex
from cdr.tex import fmt_num, fmt_int, fmt_p, fmt_ci

R = lambda name: pd.read_csv(RESULTS / name, low_memory=False)

registry = []
def register(artifact_id, artifact_type, main_or_si, latex_label, claim_id, family, notebook, result_object, dataset, raw_family, output_path, role, necessity):
    registry.append({
        "artifact_id": artifact_id, "artifact_type": artifact_type, "main_or_si": main_or_si,
        "latex_label": latex_label, "latex_reference": ("\\ref{" + latex_label + "}") if latex_label else "",
        "claim_or_threat_id": claim_id, "evidence_family": family, "source_notebook": notebook,
        "source_result_object": result_object, "source_analytical_dataset": dataset,
        "raw_source_family": raw_family, "output_path": output_path,
        "generated_during_current_run": "yes", "validation_status": "PENDING",
        "scientific_role": role, "necessity_reason": necessity,
    })

## Manual linkage validation: performer and composition distinctions

The July 2026 sample contains 200 links (191 distinct Billboard song–artist
records), sampled from 11,446 retained links. This block quantifies validation
labels and describes the evidence behind unconfirmed cases. It does not filter
the analytical linkage or change estimation samples.

`camila_linkage_adjudicated.csv` preserves the original review and the four
previously reconciled labels. The separate manual input
`data/validation/camila_linkage_error_annotations.csv` records a typology based
on the supplied screenshots for the 15 links labeled incorrect and 5 ambiguous.
The other 180 links have not been re-reviewed for this typology. Screenshot
hashes bind annotations to the evidence recorded in the reconciled review.

An explicit alternative performer can establish a distinction between a
composition credit and the linked song–artist record. An expanded credit that
includes the linked artist does not, by itself, establish a cover or false link.
Neither title/performer agreement nor this partial typology verifies an exact
master recording. The labels remain unchanged, including cases where the notes
identify uncertainty about the classification rule. No composition-level
precision or recall is inferred.


In [3]:
# Manual judgments are inputs. Joins and all reported counts are recomputed here.
validation_dir = paths.PROJECT_ROOT / "outputs/data/validation"
review = pd.read_csv(validation_dir / "camila_linkage_adjudicated.csv", dtype=str, keep_default_na=False)
annotations = pd.read_csv(paths.PROJECT_ROOT / "data/validation/camila_linkage_error_annotations.csv",
                          dtype=str, keep_default_na=False)
check(review["sample_id"].is_unique and annotations["sample_id"].is_unique,
      "manual review and evidence annotations have unique sample IDs")
check(set(annotations["sample_id"]) == set(review.loc[review["clasificacion_adjudicada"].ne("confirmado"), "sample_id"]),
      "the typology covers exactly the unconfirmed review records")
evidence_hash = review.set_index("sample_id")["captura_sha256"]
check(annotations["sample_id"].map(evidence_hash).eq(annotations["captura_sha256"]).all(),
      "annotation evidence hashes match the preserved review")
typed_review = review.merge(annotations.drop(columns="captura_sha256"), on="sample_id",
                            how="left", validate="one_to_one", sort=False)
for column in annotations.columns.difference(["sample_id", "captura_sha256"]):
    typed_review[column] = typed_review[column].fillna("no_evaluado")
check(typed_review[review.columns].equals(review), "all original review columns and labels are preserved")
display(typed_review)
typed_review.to_csv(validation_dir / "camila_linkage_error_types.csv", index=False)

classification_summary = pd.concat([
    review[column].value_counts().rename_axis("classification").reset_index(name="n").assign(review_stage=stage)
    for column, stage in [("clasificacion_camila", "original"), ("clasificacion_adjudicada", "reconciled")]
], ignore_index=True)
classification_summary["percent_of_sample"] = 100 * classification_summary["n"] / len(review)
display(classification_summary)
classification_summary.to_csv(validation_dir / "camila_linkage_classification_summary.csv", index=False)

type_summary = (typed_review.groupby(["clasificacion_adjudicada", "tipo_discrepancia"], sort=True)
                .size().reset_index(name="n"))
type_summary["percent_within_classification"] = 100 * type_summary["n"] / type_summary.groupby("clasificacion_adjudicada")["n"].transform("sum")
type_summary["percent_of_sample"] = 100 * type_summary["n"] / len(review)
display(type_summary)
type_summary.to_csv(validation_dir / "camila_linkage_error_type_summary.csv", index=False)

incorrect = typed_review.loc[typed_review["clasificacion_adjudicada"].eq("incorrecto")]
other_performer_n = incorrect["relacion_interprete"].eq("otro_interprete_explicito").sum()
print(f"{other_performer_n}/{len(incorrect)} links labeled incorrect have an explicit alternative performer.")
print(f"Sampling unit: {len(review)} links, {len(review[['song_billboard', 'artist_billboard']].drop_duplicates())} distinct Billboard records.")
print("The typology annotates existing labels and does not change analytical data or model estimates.")


PASS: manual review and evidence annotations have unique sample IDs
PASS: the typology covers exactly the unconfirmed review records
PASS: annotation evidence hashes match the preserved review
PASS: all original review columns and labels are preserved


,sample_id,link_id,song_billboard,artist_billboard,song_imdb,artist_imdb,movie_title,movie_year,movie_id,imdb_url,match_type,evidence_url_1,evidence_url_2,nota,fecha_revision,revisora,clasificacion_camila,clasificacion_adjudicada,adjudicacion,motivo_adjudicacion,fecha_adjudicacion,responsable_adjudicacion,captura,captura_sha256,revision_tipo_estado,interprete_en_captura,artista_billboard_en_credito_performer,credito_autoral_en_captura,evidencia_composicion,relacion_interprete,identidad_grabacion_maestra,nota_tipo,cuestion_de_clasificacion,fecha_revision_tipo,tipo_discrepancia
0,S001,d1ff7014274ee8beaeb1,Total Eclipse Of The Heart,Bonnie Tyler,Total Eclipse of the Heart,Bonnie Tyler,Gloria Bell,2018,6902696,https://www.imdb.com/title/tt6902696/,case_only,https://pro.imdb.com/title/tt6902696/details,https://www.youtube.com/watch?v=rxeLPFEMPkg,,2026-07-21,Camila,confirmado,confirmado,original_classification_retained,,,,Inputs_Camila_Julio2026/Evidencia/S001.png,42d7e7fab7dc2144b5f12761418ee46ec76b2e435b44cd...,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado
1,S002,329fe1775411717e658f,I Don't Care,Fall Out Boy,I Don't Care,Fall Out Boy,Sex Drive,2008,1135985,https://www.imdb.com/title/tt1135985/,exact_string,https://pro.imdb.com/title/tt1135985/details,,,2026-07-21,Camila,confirmado,confirmado,original_classification_retained,,,,Inputs_Camila_Julio2026/Evidencia/S002.png,6633ce88588485c976bd0af7a7750bd6fe93e2642ec671...,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado
2,S003,983e159f55c9ece517d9,Sadeness Part 1,Enigma,Sadeness - Part 1,Enigma,Spin Me Round,2022,14596320,https://www.imdb.com/title/tt14596320/,normalized_only,https://pro.imdb.com/title/tt14596320/details,,,2026-07-21,Camila,confirmado,confirmado,original_classification_retained,,,,Inputs_Camila_Julio2026/Evidencia/S003.png,9b6f203d4492155d7ffbb7c0e768a055fec6448e30691c...,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado
3,S004,feb13b7f305c36431e4f,You Sexy Thing,Hot Chocolate,You Sexy Thing,Hot Chocolate,Nick and Norah's Infinite Playlist,2008,981227,https://www.imdb.com/title/tt0981227/,exact_string,https://pro.imdb.com/title/tt0981227/details,,,2026-07-21,Camila,confirmado,confirmado,original_classification_retained,,,,Inputs_Camila_Julio2026/Evidencia/S004.png,0c00e1e2ddcbf878eb5335bace9a4992681ef1677e55ed...,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado
4,S005,12f6a87516a142c086f8,Rhinestone Cowboy,Glen Campbell,Rhinestone Cowboy,Glen Campbell,Irresistible,2020,9076562,https://www.imdb.com/title/tt9076562/,exact_string,https://pro.imdb.com/title/tt9076562/details,,,2026-07-21,Camila,confirmado,confirmado,original_classification_retained,,,,Inputs_Camila_Julio2026/Evidencia/S005.png,f58dd7b25a08daac3da1106b14dac785bb6afeda9c3c9b...,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,S196,9cd45b4f2f8ebe8ad18b,I Got You Babe,Sonny & Cher,I Got You Babe,Sonny & Cher,Good Times,1967,61720,https://www.imdb.com/title/tt0061720/,exact_string,https://pro.imdb.com/title/tt0061720/details,,,2026-07-26,Camila,confirmado,confirmado,original_classification_retained,,,,Inputs_Camila_Julio2026/Evidencia/S196.png,b1e8eca877acf010d6f03536ff5fae424990bdf1d3fc80...,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado,no_evaluado
196,S197,ada840940b490bf4fbc9,Grazing In The Grass,Hugh Masekela,Grazing in the Grass,Hugh Masekela,Talk to Me,2007,796368,https://www.

,classification,n,review_stage,percent_of_sample
0,confirmado,180,original,90.0
1,incorrecto,16,original,8.0
2,ambiguo,4,original,2.0
3,confirmado,180,reconciled,90.0
4,incorrecto,15,reconciled,7.5
5,ambiguo,5,reconciled,2.5


,clasificacion_adjudicada,tipo_discrepancia,n,percent_within_classification,percent_of_sample
0,ambiguo,compositor_sin_interprete_identificado,1,20.000000,0.5
1,ambiguo,credito_ampliado_incluye_artista,1,20.000000,0.5
2,ambiguo,otro_interprete_composicion_respaldada,1,20.000000,0.5
3,ambiguo,otro_interprete_con_original_alegado_no_verifi...,1,20.000000,0.5
4,ambiguo,solista_banda_no_resuelto,1,20.000000,0.5
5,confirmado,no_evaluado,180,100.000000,90.0
6,incorrecto,credito_ampliado_incluye_artista,1,6.666667,0.5
7,incorrecto,otro_interprete_composicion_respaldada,14,93.333333,7.0


14/15 links labeled incorrect have an explicit alternative performer.
Sampling unit: 200 links, 191 distinct Billboard records.
The typology annotates existing labels and does not change analytical data or model estimates.


## Interpreting the lower pre-film excess-attention criterion

This diagnostic reconstructs the existing selection from stored expected-attention
values. It does not refit curves, rematch controls, or estimate effects. The
sampling frame for this subanalysis is the film-linked group with observed
pre/post support, distinct from the broad matched comparisons with thousands
of songs. Exact Billboard record identity and the selection functions are the
same as in notebook 03. A positive cutoff means the lower tercile can contain
songs above the expected curve. Chart presence is historical, and the selection
does not require a prior attention decline or an absolute popularity threshold.


In [4]:
from cdr import dormant as dm

audit_panel = pd.read_csv(paths.OUT_DATA / "spotify_song_snapshot_excess_attention.csv", low_memory=False)
audit_master = pd.read_csv(paths.OUT_DATA / "master_song_panel.csv", low_memory=False)
audit_panel["song_artist_id"] = audit_panel["song"].astype(str) + "||" + audit_panel["artist"].astype(str)
audit_first_year = (audit_master.assign(record_id=audit_master["song"].astype(str) + "||" + audit_master["artist"].astype(str))
                    .query("film_linked == 1").dropna(subset=["first_year"]).set_index("record_id")["first_year"])
audit_wide = dm.build_snapshot_wide(audit_panel)
audit_base = dm.prepare_dormant_base(audit_wide, None, audit_first_year)
audit_dormant = audit_base["treated_dormant"].copy()
display(audit_dormant)
audit_dormant.to_csv(validation_dir / "dormancy_definition_records.csv", index=False)
definition_diagnostic = pd.DataFrame([
    {"quantity": "eligible film-linked records with pre/post support", "value": len(audit_base["treated"])},
    {"quantity": "lower-tercile records", "value": len(audit_dormant)},
    {"quantity": "pre-film excess-attention cutoff", "value": audit_base["q33"]},
    {"quantity": "lower-tercile records below expected attention", "value": audit_dormant["baseline_excess"].lt(0).sum()},
    {"quantity": "lower-tercile records at or above expected attention", "value": audit_dormant["baseline_excess"].ge(0).sum()},
    {"quantity": "median baseline Spotify popularity", "value": audit_dormant["baseline_popularity"].median()},
    {"quantity": "minimum baseline Spotify popularity", "value": audit_dormant["baseline_popularity"].min()},
    {"quantity": "maximum baseline Spotify popularity", "value": audit_dormant["baseline_popularity"].max()},
    {"quantity": "baseline Spotify popularity above 50", "value": audit_dormant["baseline_popularity"].gt(50).sum()},
    {"quantity": "minimum baseline age since Billboard debut", "value": audit_dormant["baseline_age"].min()},
])
display(definition_diagnostic)
definition_diagnostic.to_csv(validation_dir / "dormancy_definition_diagnostic.csv", index=False)
saved_support = R("dormant_definition_support.csv")["n"].to_numpy()
check(np.allclose(saved_support, [len(audit_base["treated"]), round(audit_base["q33"], 3), len(audit_dormant)]),
      "the diagnostic exactly reproduces the existing dormant support")
print("The diagnostic describes the original selection. No analytical record or effect estimate has changed.")


,song_artist_id,event_year,film_linked,baseline_snapshot,first_post_snapshot,baseline_popularity,first_post_popularity,baseline_age,first_post_age,baseline_excess,first_post_excess,excess_change,chart_year,weeks,peak_pos,artist_catalog_size_excl_song,artist_total_weeks_excl_song,artist_superstar_top1pct
2,(You Gotta) Fight For Your Right (To Party!)||...,2021.0,1,2017-07,2022-08,29.0,36.0,31.55,36.65,-0.728199,1.680718,2.408917,1986.0,18.0,7.0,5.0,55.0,0.0
6,5.7.0.5.||City Boy,2017.0,1,2016-10,2017-07,5.0,9.0,38.80,39.55,-17.125264,-15.636048,1.489216,1978.0,12.0,27.0,0.0,0.0,0.0
14,Al Di La||Connie Francis,2021.0,1,2017-07,2022-08,23.0,35.0,54.55,59.65,5.678894,14.453171,8.774278,1963.0,5.0,90.0,52.0,482.0,1.0
15,Al Di La'||Emilio Pericoli,2019.0,1,2017-07,2022-08,21.0,40.0,55.55,60.65,4.080952,19.927885,15.846933,1962.0,14.0,6.0,0.0,0.0,0.0
18,Alright||Kendrick Lamar,2018.0,1,2017-07,2022-08,72.0,79.0,2.55,7.65,9.806534,19.729391,9.922858,2015.0,14.0,81.0,39.0,306.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
338,Raindrops||Dee Clark,2018.0,1,2017-07,2022-08,19.0,31.0,56.55,61.65,2.473677,11.393122,8.919445,1961.0,16.0,2.0,9.0,96.0,0.0
339,Riot||Hugh Masekela,2020.0,1,2017-07,2022-08,9.0,46.0,48.55,53.65,-10.942233,22.399179,33.341413,1969.0,8.0,55.0,3.0,25.0,0.0
346,T-Shirt||Migos,2018.0,1,2017-07,2022-08,85.0,66.0,0.55,5.65,8.412185,4.845303,-3.566881,2017.0,25.0,19.0,16.0,86.0,0.0
347,That's My Girl||Fifth Harmony,2018.0,1,2017-07,2022-08,76.0,68.0,1.55,6.65,8.662218,7.786533,-0.875685,2016.0,3.0,73.0,3.0,41.0,0.0


,quantity,value
0,eligible film-linked records with pre/post sup...,355.000000
1,lower-tercile records,119.000000
2,pre-film excess-attention cutoff,9.876266
3,lower-tercile records below expected attention,42.000000
4,lower-tercile records at or above expected att...,77.000000
5,median baseline Spotify popularity,32.000000
6,minimum baseline Spotify popularity,0.000000
7,maximum baseline Spotify popularity,85.000000
8,baseline Spotify popularity above 50,39.000000
9,minimum baseline age since Billboard debut,0.550000


PASS: the diagnostic exactly reproduces the existing dormant support
The diagnostic describes the original selection. No analytical record or effect estimate has changed.


## Artifact registry

One metadata row per retained artifact: claim mapping, generating notebook, source result objects, and output path. Generation happens in notebooks 01–05.

In [5]:
register("data_audit_table", "table", "main", "tab:data-audit", "C1", "data construction and linkage", "00-01", "master_support_counts.csv; movie_metadata_coverage.csv", "master_song_panel.csv", "Billboard/Spotify/IMDb/Last.fm raw inputs", "outputs/tables/main/data_audit_table.tex", "documents the analytic universe and support counts behind every analysis", "single authoritative statement of the support counts used throughout the paper")
register("table_si_platform_coverage", "table", "si", "tab:si-platform-coverage", "C2", "data construction and linkage", "01", "platform_coverage.csv", "master_song_panel.csv", "Spotify snapshots + Last.fm", "outputs/tables/si/table_si_platform_coverage.tex", "documents per-platform and per-snapshot linkage support", "only artifact recording snapshot-level linkage support; no other retained artifact carries it")
register("table_si_movie_metadata_coverage", "table", "si", "tab:si-movie-metadata-coverage", "C3", "data construction and linkage", "01", "movie_metadata_coverage.csv", "master_song_panel.csv", "IMDb + Box Office Mojo (thesis lineage)", "outputs/tables/si/table_si_movie_metadata_coverage.tex", "single coverage/missingness object for all movie-level variables", "consolidates all movie-metadata support counts in one table with one denominator")
register("figure1_attention_regimes_matching", "figure", "main", "fig:attention-matching", "C5;C11;C12", "platform coverage and attention differences; matched song comparisons", "02", "standardized_premiums.csv; song_matching_estimates.csv", "spotify_song_snapshot_panel.csv; lastfm_song_panel.csv", "Spotify snapshots + Last.fm + Billboard", "outputs/figures/main/figure1_attention_regimes_matching.pdf", "cross-platform standardized premiums, matched premiums, and balance summary", "the paper's first-result figure combining attention differences, matching, and balance evidence")
register("table_si_attention_premiums", "table", "si", "tab:si-attention-premiums", "C5", "platform coverage and attention differences", "02", "standardized_premiums.csv; attention_premiums.csv", "spotify_song_snapshot_panel.csv; lastfm_song_panel.csv", "Spotify snapshots + Last.fm", "outputs/tables/si/table_si_attention_premiums.tex", "raw-unit and standardized premiums behind the main attention-differences figure", "one consolidated table replaces separate raw Spotify, standardized, and Last.fm premium tables")
register("table_si_song_matching_estimates", "table", "si", "tab:si-song-matching-estimates", "C11;C12", "matched song-level comparisons", "02", "song_matching_estimates.csv", "spotify_song_snapshot_panel.csv", "Spotify snapshots + Billboard", "outputs/tables/si/table_si_song_matching_estimates.tex", "full matched support and estimates behind the main matching panels", "carries per-snapshot support counts and balance summaries that the main figure reports only graphically")
register("table_si_song_matching_inference", "table", "si", "tab:si-song-matching-inference", "C14", "matched song-level comparisons", "02", "song_matching_inference.csv", "spotify_song_snapshot_panel.csv", "Spotify snapshots + Billboard", "outputs/tables/si/table_si_song_matching_inference.tex", "quantifies control reuse and replacement-aware uncertainty for the song-level matched designs", "control reuse is a real inferential threat for with-replacement matching; no other artifact carries it")
register("table_memory_decay_key_results", "table", "main", "tab:memory-decay-key-results", "C9", "expected-attention baselines", "03", "excess_premiums.csv; matched_excess.csv; matched_excess_inference.csv; temporal_excess.csv; decay_equivalent_audit.csv; dormant_integrated_inference.csv", "spotify_song_snapshot_excess_attention.csv", "Spotify snapshots + Billboard + IMDb", "outputs/tables/main/table_memory_decay_key_results.tex", "summarizes broad excess-attention levels, prior-attention matched differences, and temporal contrasts", "single main-text table carrying the paper's central excess-attention estimates")
register("figure2_expected_memory_excess_dormant", "figure", "main", "fig:excess-memory", "C7;C9;C10;C14", "expected-attention baselines; lower pre-film excess-attention reactivation", "03", "memorydecay_age_means.csv; memorydecay_fits.csv; excess_premiums.csv; matched_excess.csv; matched_excess_inference.csv; temporal_excess.csv; decay_equivalent_age_grid.csv; dormant_assignment_distribution.csv; dormant_integrated_inference.csv", "spotify_song_snapshot_excess_attention.csv", "Spotify snapshots + Billboard + IMDb", "outputs/figures/main/figure2_expected_memory_excess_dormant.pdf", "expected-attention curves, level differences, temporal contrasts, decay-equivalent translations, and integrated inference", "the paper's central reactivation figure")
register("table_si_memorydecay_support", "table", "si", "tab:si-memorydecay-support", "C7", "expected-attention baselines", "03", "memorydecay_support.csv", "spotify_song_snapshot_excess_attention.csv", "Spotify snapshots + Billboard", "outputs/tables/si/table_si_memorydecay_support.tex", "documents the aggregation support behind every fitted baseline", "fit validity cannot be audited without the never-film cell support")
register("table_si_memorydecay_fits", "table", "si", "tab:si-memorydecay-fits", "C7;C8", "expected-attention baselines", "03", "memorydecay_fits.csv; memorydecay_model_comparison.csv", "spotify_song_snapshot_excess_attention.csv", "Spotify snapshots + Billboard", "outputs/tables/si/table_si_memorydecay_fits.tex", "fitted parameters, degeneracy flags, and quantitative model comparison", "the expected-value baseline is unauditable without the fitted parameters; comparison columns document the canonical-model choice")
register("table_si_decay_equivalent_audit", "table", "si", "tab:si-decay-equivalent-audit", "C10", "expected-attention baselines", "03", "decay_equivalent_audit.csv; dormant_integrated_inference.csv", "spotify_song_snapshot_excess_attention.csv", "Spotify snapshots + Billboard", "outputs/tables/si/table_si_decay_equivalent_audit.tex", "documents translation support, dispersion, and inversion validity for every decay-equivalent value in the paper", "decay-equivalent numbers are uninterpretable without their estimability diagnostics")
register("table_si_matching_balance", "table", "si", "tab:si-matching-balance", "C13", "matched song-level comparisons", "02-03", "song_matching_balance_long.csv; strict_temporal_balance.csv; dormant_reduced_form_balance.csv", "spotify_song_snapshot_panel.csv", "Spotify snapshots + Billboard", "outputs/tables/si/table_si_matching_balance.tex", "covariate-level balance transparency for every song-level matched design", "one long-format artifact replaces the per-snapshot-per-design balance table explosion")
register("table_si_matched_excess_inference", "table", "si", "tab:si-matched-excess-inference", "C14", "matched song-level comparisons", "03", "matched_excess_inference.csv", "spotify_song_snapshot_excess_attention.csv", "Spotify snapshots + Billboard", "outputs/tables/si/table_si_matched_excess_inference.tex", "replacement-aware uncertainty for the pooled matched-excess premium reported in the main text", "the main matched-excess interval rests on this dependence-aware inference; no other retained artifact documents it")
register("table_si_dormant_definitions", "table", "si", "tab:si-dormant-definitions", "C15", "lower pre-film excess-attention-song reactivation", "03", "dormant_reduced_form_placebo.csv", "master_song_panel.csv", "Spotify snapshots + Billboard + IMDb", "outputs/tables/si/table_si_dormant_definitions.tex", "shows the lower pre-film excess-attention signature is not specific to the primary excess-attention definition", "one consolidated table replaces the separate definition-support, results, and bootstrap tables")
register("table_si_dormant_integrated_inference", "table", "si", "tab:si-dormant-integrated-inference", "C14;C16", "lower pre-film excess-attention-song reactivation", "03", "dormant_integrated_inference.csv; dormant_definition_support.csv", "data/inference replication files; spotify_song_snapshot_excess_attention.csv", "stored replication-level inference + Spotify snapshots", "outputs/tables/si/table_si_dormant_integrated_inference.tex", "complete audit of the primary integrated lower pre-film excess-attention inference and its support", "the primary lower pre-film excess-attention design is unauditable without assignment stability, bootstrap, exit, translation, reuse, and inversion diagnostics in one place")
register("figure3_embedding_visibility_temporal", "figure", "main", "fig:embedding-temporal", "C17;C18;C19;C20", "embedding and visibility; temporal consistency", "04", "repeated_embedding.csv; repeated_embedding_by_snapshot.csv; visibility.csv; visibility_terciles.csv; event_time.csv; strict_temporal_pooled.csv", "spotify_song_snapshot_panel.csv; snapshot_time_respecting_exposures.csv", "Spotify snapshots + IMDb + box office", "outputs/figures/main/figure3_embedding_visibility_temporal.pdf", "time-respecting embedding and visibility gradients (terciles visualize shape; the continuous model is the primary visibility estimate, reported in the caption) with temporal consistency checks", "the paper's cue-intensity and temporal-evidence figure")
register("table_si_repeated_embedding", "table", "si", "tab:si-repeated-embedding", "C17", "embedding and visibility", "04", "repeated_embedding.csv", "spotify_song_snapshot_panel.csv; snapshot_time_respecting_exposures.csv; lastfm_song_panel.csv", "Spotify snapshots + IMDb + Last.fm", "outputs/tables/si/table_si_repeated_embedding.tex", "full embedding-gradient evidence including the future-information robustness comparison", "the lifetime-versus-realized comparison addresses the reverse-timing threat and appears nowhere else")
register("table_si_visibility_models", "table", "si", "tab:si-visibility-models", "C18", "embedding and visibility", "04", "visibility.csv", "spotify_song_snapshot_panel.csv; snapshot_time_respecting_exposures.csv", "Spotify snapshots + IMDb + box office", "outputs/tables/si/table_si_visibility_models.tex", "exact specification and support behind the main visibility coefficient", "documents the primary continuous visibility model's inference and support")
register("table_si_visibility_selection", "table", "si", "tab:si-visibility-selection", "C18", "embedding and visibility", "04", "visibility_selection.csv; visibility.csv", "spotify_song_snapshot_panel.csv; snapshot_time_respecting_exposures.csv", "Spotify snapshots + IMDb + box office", "outputs/tables/si/table_si_visibility_selection.tex", "documents realized-box-office availability within the time-respecting film-linked universe; the realized row equals the continuous-model support exactly", "box-office missingness is a distinct inferential threat to the visibility gradient")
register("table_si_temporal_event_time", "table", "si", "tab:si-temporal-event-time", "C19", "temporal consistency", "04", "event_time.csv; event_time_support.csv", "spotify_song_snapshot_panel.csv", "Spotify snapshots + Billboard + IMDb", "outputs/tables/si/table_si_temporal_event_time.tex", "documents pre-event selection and post-event patterns with per-bin support", "the selection-into-placement threat is documented by the pre-event bin and its support")
register("table_si_strict_temporal", "table", "si", "tab:si-strict-temporal", "C20", "temporal consistency", "03-04", "strict_temporal_pooled.csv; strict_temporal_effects.csv", "master_song_panel.csv", "Spotify snapshots + Billboard + IMDb", "outputs/tables/si/table_si_strict_temporal.tex", "support, cohort decomposition, and randomization inference for the conservative temporal check", "the 93-pair design's cohort structure and randomization inference appear nowhere else")
register("figure4_artist_catalog_regimes", "figure", "main", "fig:catalog", "C21;C23;C24", "artist-catalog attention", "05", "catalog_matching_estimates.csv; catalog_matching_inference.csv; catalog_retention_by_cohort.csv; catalog_retention_pooled.csv", "spotify_song_snapshot_panel.csv", "Spotify snapshots + Billboard + IMDb", "outputs/figures/main/figure4_artist_catalog_regimes.pdf", "attenuation of the matched catalog premium under prior-attention control and the matched attention change diagnostic with artist-level replacement-aware intervals", "shows attenuation after prior-attention matching and inconclusive matched changes among other Billboard songs by the same artists")
register("table_si_catalog_static", "table", "si", "tab:si-catalog-static", "C21", "artist-catalog attention", "05", "catalog_static.csv", "spotify_song_snapshot_panel.csv; lastfm_song_panel.csv", "Spotify snapshots + Last.fm + Billboard", "outputs/tables/si/table_si_catalog_static.tex", "single authoritative static catalog regression table across platforms and snapshots", "one table replaces the duplicated static-catalog tables of the previous SI")
register("table_si_catalog_dynamic", "table", "si", "tab:si-catalog-dynamic", "C22", "artist-catalog attention", "05", "catalog_dynamic.csv; catalog_dynamic_support.csv", "spotify_song_snapshot_panel.csv", "Spotify snapshots + Billboard + IMDb", "outputs/tables/si/table_si_catalog_dynamic.tex", "documents catalog-level selection (positive pre-event premium) with support", "the catalog selection threat is carried by this table alone")
register("table_si_catalog_matching", "table", "si", "tab:si-catalog-matching", "C23", "artist-catalog attention", "05", "catalog_matching_estimates.csv", "spotify_song_snapshot_panel.csv; lastfm_song_panel.csv", "Spotify snapshots + Last.fm + Billboard", "outputs/tables/si/table_si_catalog_matching.tex", "matched catalog estimates and support behind the main catalog figure", "carries per-snapshot matched support the main figure reports only graphically")
register("table_si_catalog_balance", "table", "si", "tab:si-catalog-balance", "C23", "artist-catalog attention", "05", "catalog_matching_balance_long.csv", "spotify_song_snapshot_panel.csv", "Spotify snapshots + Billboard", "outputs/tables/si/table_si_catalog_balance.tex", "covariate-level balance transparency for every catalog matched design", "one long-format artifact replaces the catalog balance-table explosion")
register("table_si_catalog_matching_inference", "table", "si", "tab:si-catalog-matching-inference", "C23", "artist-catalog attention", "05", "catalog_matching_inference.csv", "spotify_song_snapshot_panel.csv", "Spotify snapshots + Billboard", "outputs/tables/si/table_si_catalog_matching_inference.tex", "replacement-aware qualification of the matched catalog premium", "the near-zero August 2022 qualification in the main text rests on this table")
register("figure_si_catalog_event_time_selection", "figure", "si", "fig:si-catalog-event-time-selection", "C22", "artist-catalog attention", "05", "catalog_dynamic.csv", "spotify_song_snapshot_panel.csv", "Spotify snapshots + Billboard + IMDb", "outputs/figures/si/figure_si_catalog_event_time_selection.pdf", "shows that the catalog premium predates the observed event; the positive pre-event coefficient is a selection diagnostic, not an effect", "carries the pre-event selection evidence that qualifies the catalog analyses now that the main figure reports the attention-change diagnostic")
register("table_si_catalog_retention", "table", "si", "tab:si-catalog-retention", "C24", "artist-catalog attention", "05", "catalog_retention_by_cohort.csv; catalog_retention_pooled.csv", "spotify_song_snapshot_panel.csv", "Spotify snapshots + Billboard + IMDb", "outputs/tables/si/table_si_catalog_retention.tex", "cohort-level direct attention-change estimates with artist-level replacement-aware inference and the pooled descriptive summary", "the main attention-change diagnostic (Figure 4B) rests on these values; no other artifact documents the artist-level inference")
register("table_si_catalog_retention_loco", "table", "si", "tab:si-catalog-retention-loco", "C24", "artist-catalog attention", "05", "catalog_retention_leave_one_cohort_out.csv", "spotify_song_snapshot_panel.csv", "Spotify snapshots + Billboard + IMDb", "outputs/tables/si/table_si_catalog_retention_loco.tex", "shows the pooled attention-change summary is not driven by a single cohort and remains inconclusive in every leave-one-out combination", "guards the pooled descriptive summary against single-cohort driving")
register("table_si_cluster_inference", "table", "si", "tab:si-cluster-inference", "C24", "inference and robustness", "02;04;05", "pooled_premium_cluster_rows.csv; embedding_visibility_cluster_rows.csv; robustness_cluster_rows.csv", "spotify_song_snapshot_panel.csv", "Spotify snapshots + Billboard", "outputs/tables/si/table_si_cluster_inference.tex", "documents that repeated song-snapshot dependence does not change any pooled conclusion", "song-snapshot dependence is a distinct inferential threat for every pooled model")
register("table_si_robustness_summary", "table", "si", "tab:si-robustness-summary", "C25", "inference and robustness", "05", "robustness_cluster_rows.csv", "spotify_song_snapshot_panel.csv", "Spotify snapshots + Billboard", "outputs/tables/si/table_si_robustness_summary.tex", "one consolidated robustness summary addressing selection, superstar, and catalog-definition threats", "replaces the overlapping placebo, superstar-exclusion, and alternative-definition tables")
register("figure_si_crossplatform_embedding", "figure", "si", "fig:si-crossplatform-embedding", "C17;C6", "embedding and visibility", "04;05", "repeated_embedding.csv", "spotify_song_snapshot_panel.csv; lastfm_song_panel.csv", "Spotify snapshots + Last.fm + IMDb", "outputs/figures/si/figure_si_crossplatform_embedding.pdf", "monotonic repeated-embedding gradient on both platforms, with time-respecting Spotify exposure and lifetime single-snapshot Last.fm exposure", "reveals cross-platform concordance of the monotonic embedding pattern while preserving the different timing semantics of the two platforms; the shape comparison is not readable from separate table rows")
register("figure_si_dormant_rtm", "figure", "si", "fig:si-dormant-rtm", "C15", "lower pre-film excess-attention-song reactivation", "03", "dormant_reduced_form_placebo.csv", "master_song_panel.csv", "Spotify snapshots + Billboard + IMDb", "outputs/figures/si/figure_si_dormant_rtm.pdf", "raw treated renewal versus matched pseudo-event control renewal and the placebo-adjusted residual across dormancy definitions", "makes the regression-to-the-mean structure visible: it reveals the attenuation from raw renewal to matched-control renewal and the remaining positive film-linked increment across well-powered definitions, which is difficult to read from the table alone")
register("figure_si_memorydecay_model_comparison", "figure", "si", "fig:si-memorydecay-model-comparison", "C7;C8", "expected-attention baselines", "03", "memorydecay_age_means.csv; memorydecay_fits.csv", "spotify_song_snapshot_excess_attention.csv", "Spotify snapshots + Billboard", "outputs/figures/si/figure_si_memorydecay_model_comparison.pdf", "canonical versus exponential and log-normal fits on never-film age means for every snapshot", "the canonical-model choice is a theoretically important modeling decision requiring visual diagnostics")
register("figure_si_dormant_pseudoevent", "figure", "si", "fig:si-dormant-pseudoevent", "C14", "lower pre-film excess-attention-song reactivation", "03", "data/inference/dormant_pseudoevent_monte_carlo_replications.csv", "spotify_song_snapshot_excess_attention.csv", "stored replication-level inference", "outputs/figures/si/figure_si_dormant_pseudoevent.pdf", "distribution of all 500 assignment-specific lower pre-film excess-attention contrasts", "pseudo-event integration is part of the primary design; its stability distribution is not visible in any table row alone")
register("figure_si_lastfm_age_profile", "figure", "si", "fig:si-lastfm-age-profile", "C6", "platform coverage and attention differences", "05", "lastfm_age_profile.csv", "lastfm_song_panel.csv", "Last.fm + Billboard", "outputs/figures/si/figure_si_lastfm_age_profile.pdf", "cumulative listener reach by song age showing the non-monotone young-age profile", "justifies excluding Last.fm from expected-attention estimation; no table shows the profile shape")


## Artifact registry


In [6]:
trace = pd.DataFrame(registry)
missing_files = [r["output_path"] for _, r in trace.iterrows() if not (paths.PROJECT_ROOT / r["output_path"]).exists()]
check(not missing_files, f"every registered artifact file exists ({len(trace)} artifacts)")
trace["validation_status"] = "PASS"
display(trace)
show_and_save_table(trace[["artifact_id", "artifact_type", "main_or_si", "latex_label", "claim_or_threat_id", "evidence_family", "source_notebook", "validation_status"]], paths.OUT_DATA / "artifact_trace_preview.csv")
trace.to_csv(paths.OUT_DATA / "artifact_trace.csv", index=False)
print(f"artifact_trace.csv written: {len(trace)} retained artifacts "
      f"({(trace['main_or_si'].eq('main')).sum()} main, {(trace['main_or_si'].eq('si')).sum()} SI)")
check(trace["claim_or_threat_id"].str.len().gt(0).all(), "every retained artifact maps to at least one claim or inferential threat")


PASS: every registered artifact file exists (38 artifacts)


,artifact_id,artifact_type,main_or_si,latex_label,latex_reference,claim_or_threat_id,evidence_family,source_notebook,source_result_object,source_analytical_dataset,raw_source_family,output_path,generated_during_current_run,validation_status,scientific_role,necessity_reason
0,data_audit_table,table,main,tab:data-audit,\ref{tab:data-audit},C1,data construction and linkage,00-01,master_support_counts.csv; movie_metadata_cove...,master_song_panel.csv,Billboard/Spotify/IMDb/Last.fm raw inputs,outputs/tables/main/data_audit_table.tex,yes,PASS,documents the analytic universe and support co...,single authoritative statement of the support ...
1,table_si_platform_coverage,table,si,tab:si-platform-coverage,\ref{tab:si-platform-coverage},C2,data construction and linkage,01,platform_coverage.csv,master_song_panel.csv,Spotify snapshots + Last.fm,outputs/tables/si/table_si_platform_coverage.tex,yes,PASS,documents per-platform and per-snapshot linkag...,only artifact recording snapshot-level linkage...
2,table_si_movie_metadata_coverage,table,si,tab:si-movie-metadata-coverage,\ref{tab:si-movie-metadata-coverage},C3,data construction and linkage,01,movie_metadata_coverage.csv,master_song_panel.csv,IMDb + Box Office Mojo (thesis lineage),outputs/tables/si/table_si_movie_metadata_cove...,yes,PASS,single coverage/missingness object for all mov...,consolidates all movie-metadata support counts...
3,figure1_attention_regimes_matching,figure,main,fig:attention-matching,\ref{fig:attention-matching},C5;C11;C12,platform coverage and attention differences; m...,02,standardized_premiums.csv; song_matching_estim...,spotify_song_snapshot_panel.csv; lastfm_song_p...,Spotify snapshots + Last.fm + Billboard,outputs/figures/main/figure1_attention_regimes...,yes,PASS,"cross-platform standardized premiums, matched ...",the paper's first-result figure combining atte...
4,table_si_attention_premiums,table,si,tab:si-attention-premiums,\ref{tab:si-attention-premiums},C5,platform coverage and attention differences,02,standardized_premiums.csv; attention_premiums.csv,spotify_song_snapshot_panel.csv; lastfm_song_p...,Spotify snapshots + Last.fm,outputs/tables/si/table_si_attention_premiums.tex,yes,PASS,raw-unit and standardized premiums behind the ...,one consolidated table replaces separate raw S...
5,table_si_song_matching_estimates,table,si,tab:si-song-matching-estimates,\ref{tab:si-song-matching-estimates},C11;C12,matched song-level comparisons,02,song_matching_estimates.csv,spotify_song_snapshot_panel.csv,Spotify snapshots + Billboard,outputs/tables/si/table_si_song_matching_estim...,yes,PASS,full matched support and estimates behind the ...,carries per-snapshot support counts and balanc...
6,table_si_song_matching_inference,table,si,tab:si-song-matching-inference,\ref{tab:si-song-matching-inference},C14,matched song-level comparisons,02,song_matching_inference.csv,spotify_song_snapshot_panel.csv,Spotify snapshots + Billboard,outputs/tables/si/table_si_song_matching_infer...,yes,PASS,quantifies control reuse and replacement-aware...,control reuse is a real inferential threat for...
7,table_memory_decay_key_results,table,main,tab:memory-decay-key-results,\ref{tab:memory-decay-key-results},C9,expected-attention baselines,03,excess_premiums.csv; matched_excess.csv; match...,spotify_song_snapshot_excess_attention.csv,Spotify snapshots + Billboard + IMDb,outputs/tables/main/table_memory_decay_key_res...,yes,PASS,"summarizes broad excess-attention levels, prio...",single main-text table carrying the paper's ce...
8,figure2_expected_memory_excess_dormant,figure,main,fig:excess-memory,\ref{fig:excess-memory},C7;C9;C10;C14,expected-attention baselines; lower pre-film e...,03,memorydecay_age_means.csv; memorydecay_fits.cs...,spotify_song_snapshot_excess_attention.csv,Spotify snapshots + Billboard + IMDb,outputs/figures/main/figure2_expected_memory_e...,yes,PASS,"expected-attention curves, level differences, ...",the paper's central reactivation figure
9,table_si_memorydeca

,artifact_id,artifact_type,main_or_si,latex_label,claim_or_threat_id,evidence_family,source_notebook,validation_status
0,data_audit_table,table,main,tab:data-audit,C1,data construction and linkage,00-01,PASS
1,table_si_platform_coverage,table,si,tab:si-platform-coverage,C2,data construction and linkage,01,PASS
2,table_si_movie_metadata_coverage,table,si,tab:si-movie-metadata-coverage,C3,data construction and linkage,01,PASS
3,figure1_attention_regimes_matching,figure,main,fig:attention-matching,C5;C11;C12,platform coverage and attention differences; m...,02,PASS
4,table_si_attention_premiums,table,si,tab:si-attention-premiums,C5,platform coverage and attention differences,02,PASS
5,table_si_song_matching_estimates,table,si,tab:si-song-matching-estimates,C11;C12,matched song-level comparisons,02,PASS
6,table_si_song_matching_inference,table,si,tab:si-song-matching-inference,C14,matched song-level comparisons,02,PASS
7,table_memory_decay_key_results,table,main,tab:memory-decay-key-results,C9,expected-attention baselines,03,PASS
8,figure2_expected_memory_excess_dormant,figure,main,fig:excess-memory,C7;C9;C10;C14,expected-attention baselines; lower pre-film e...,03,PASS
9,table_si_memorydecay_support,table,si,tab:si-memorydecay-support,C7,expected-attention baselines,03,PASS


artifact_trace.csv written: 38 retained artifacts (6 main, 32 SI)
PASS: every retained artifact maps to at least one claim or inferential threat


## Validation: support counts and headline results


In [7]:
dorm = R("dormant_integrated_inference.csv").set_index("quantity")["value"]
visibility = R("visibility.csv")
master = pd.read_csv(paths.OUT_DATA / "master_song_panel.csv", low_memory=False)
panel_all = pd.read_csv(paths.OUT_DATA / "spotify_song_snapshot_panel.csv", low_memory=False)
observed = {
    "billboard_universe": len(master),
    "film_linked_records": int(master["film_linked"].sum()),
    "spotify_high_coverage_observations": len(panel_all),
    "expected_memory_observations": int((panel_all["age"] >= 0).sum()),
    "lastfm_records": int(master["lastfm_listeners"].notna().sum()),
    "dormant_excess": float(dorm["dormant_excess_point_estimate"]),
    "dormant_excess_ci_low": float(dorm["dormant_excess_integrated_ci_low"]),
    "dormant_excess_ci_high": float(dorm["dormant_excess_integrated_ci_high"]),
    "dormant_exit_pp": float(dorm["exit_difference_pp"]),
    "pseudo_event_positive_assignments": float(dorm["n_positive_assignments"]),
    "decay_equivalent_years": float(dorm["dormant_decay_equivalent_years"]),
    "film_visibility_n": float(visibility.iloc[0]["n_observations"]),
}
expected = pd.read_csv(paths.DATA / "expected_results.csv")
rows = []
for _, row in expected.iterrows():
    val = observed.get(row["result_id"], np.nan)
    ok = bool(np.isfinite(val) and abs(val - float(row["expected_value"])) <= float(row["tolerance"]))
    rows.append({**row.to_dict(), "observed_value": val, "pass": ok})
report = pd.DataFrame(rows)
show_and_save_table(report, paths.OUT_DATA / "validation_results.csv")
check(report["pass"].all(), f"{int(report['pass'].sum())} / {len(report)} headline results validate against the closed expected values")

,result_id,result_name,expected_value,tolerance,observed_value,pass
0,billboard_universe,Billboard song-artist universe,32124.000000,0.00,32124.0000,True
1,film_linked_records,Film-linked Billboard records,4683.000000,0.00,4683.0000,True
2,spotify_high_coverage_observations,High-coverage Spotify song-snapshot observations,83785.000000,0.00,83785.0000,True
3,expected_memory_observations,Valid expected-memory observations,83780.000000,0.00,83780.0000,True
4,lastfm_records,Last.fm July 2017 records with listener counts,17988.000000,0.00,17988.0000,True
5,dormant_excess,Dormant excess-attention contrast,5.044000,0.02,5.0438,True
6,dormant_excess_ci_low,Dormant excess lower CI,2.971735,0.03,2.9717,True
7,dormant_excess_ci_high,Dormant excess upper CI,6.868017,0.03,6.8680,True
8,dormant_exit_pp,Dormant low-memory exit difference percentage ...,17.987000,0.05,17.9866,True
9,pseudo_event_positive_assignments,Positive pseudo-event assignments,500.000000,0.00,500.0000,True


PASS: 12 / 12 headline results validate against the closed expected values


## Validation: catalog attention-change contract

Every value reported in the rebuilt main Figure 4 (both panels) is validated against hard
closed expected values: the historical and prior-attention matched catalog premiums and
their pooled combinations (Panel A), the replacement-aware prior-attention intervals
(Panel A), and the matched attention-change estimates with their primary artist-level two-way
intervals, the pooled descriptive summary, and the leave-one-cohort-out sensitivities
(Panel B). Bootstrap outputs are deterministic under the fixed seeds, so tight tolerances
apply.


In [8]:
cat_checks = []
def cat_check(name, actual, expected_value, tol):
    ok = bool(np.isfinite(float(actual)) and abs(float(actual) - float(expected_value)) <= tol)
    cat_checks.append((name, ok))
    if not ok:
        print(f"FAIL: {name}: observed {float(actual):.4f} vs expected {expected_value} (tol {tol})")

cret_v = R("catalog_retention_by_cohort.csv").set_index("cohort")
cpool_v = R("catalog_retention_pooled.csv").iloc[0]
cloco_v = R("catalog_retention_leave_one_cohort_out.csv").set_index("omitted_cohort")
cmatch_v = R("catalog_matching_estimates.csv")
cinf_v = R("catalog_matching_inference.csv")

def match_est(design_start, snap):
    m = cmatch_v[(cmatch_v["platform"].eq("Spotify")) & (cmatch_v["design"].str.startswith(design_start)) & (cmatch_v["snapshot"].eq(snap))]
    return float(m.iloc[0]["estimate"])

for snap, npairs in [("2017-07", 6794), ("2022-08", 7712), ("2025", 9285)]:
    cat_check(f"retention pair support {snap}", cret_v.loc[snap, "n_pairs"], npairs, 0)
for snap, val in [("2017-07", 2.890), ("2022-08", 1.380), ("2025", 1.456)]:
    cat_check(f"historical matched premium {snap}", match_est("catalog historical", snap), val, 0.01)
cat_check("pooled historical matched premium", match_est("catalog historical", "pooled (inverse-variance)"), 2.044, 0.01)
for snap, val in [("2017-07", 0.457), ("2022-08", -0.309), ("2025", 0.439)]:
    cat_check(f"prior-attention matched premium {snap}", match_est("catalog baseline-attention", snap), val, 0.01)
cat_check("pooled prior-attention matched premium", match_est("catalog baseline-attention", "pooled (inverse-variance)"), 0.306, 0.01)
for snap, lo, hi in [("2017-07", 0.1144, 0.7933), ("2022-08", -0.9702, 0.2569), ("2025", 0.0896, 0.8138)]:
    r = cinf_v[(cinf_v["variant"].eq("with replacement")) & (cinf_v["snapshot"].eq(snap))].iloc[0]
    cat_check(f"prior cluster-control CI low {snap}", r["cluster_control_ci_low"], lo, 0.02)
    cat_check(f"prior cluster-control CI high {snap}", r["cluster_control_ci_high"], hi, 0.02)
for snap, dch, lo, hi in [("2017-07", 0.2726, -0.3567, 0.8895), ("2022-08", -0.4769, -1.7810, 0.7145), ("2025", 0.3961, -0.2198, 0.9506)]:
    cat_check(f"difference-in-change {snap}", cret_v.loc[snap, "difference_in_change"], dch, 0.002)
    cat_check(f"primary artist-level CI low {snap}", cret_v.loc[snap, "primary_ci_low"], lo, 0.01)
    cat_check(f"primary artist-level CI high {snap}", cret_v.loc[snap, "primary_ci_high"], hi, 0.01)
cat_check("pooled difference-in-change", cpool_v["difference_in_change"], 0.2525, 0.005)
cat_check("pooled artist-level CI low", cpool_v["ci_low"], -0.1517, 0.01)
cat_check("pooled artist-level CI high", cpool_v["ci_high"], 0.6568, 0.01)
for omit, val in [("2017-07", 0.2378), ("2022-08", 0.3376), ("2025", 0.1246)]:
    cat_check(f"leave-one-cohort-out pooled (omit {omit})", cloco_v.loc[omit, "difference_in_change"], val, 0.01)

n_pass = sum(ok for _, ok in cat_checks)
check(n_pass == len(cat_checks), f"catalog-retention validation {n_pass}/{len(cat_checks)} PASS against the closed retention values")


PASS: catalog-retention validation 32/32 PASS against the closed retention values


## Validation: LaTeX artifact references and legacy-value sweep


In [9]:
import re as _re
import hashlib as _hl
import shutil as _sh

# The manuscript compiles from its own Figures/ and Tables/ copies. Refresh
# them from the authoritative outputs before validating references, so a clean
# run can never compile stale copies.
for _, r_ in trace.iterrows():
    src_ = paths.PROJECT_ROOT / r_["output_path"]
    dst_dir_ = paths.MANUSCRIPT / ("Figures" if r_["artifact_type"] == "figure" else "Tables") / ("Main" if r_["main_or_si"] == "main" else "SI")
    dst_dir_.mkdir(parents=True, exist_ok=True)
    _sh.copy2(src_, dst_dir_ / src_.name)
check(all(_hl.sha256((paths.PROJECT_ROOT / r_["output_path"]).read_bytes()).hexdigest()
          == _hl.sha256((paths.MANUSCRIPT / ("Figures" if r_["artifact_type"] == "figure" else "Tables")
                         / ("Main" if r_["main_or_si"] == "main" else "SI")
                         / Path(r_["output_path"]).name).read_bytes()).hexdigest()
          for _, r_ in trace.iterrows()),
      "manuscript Figures/Tables copies are byte-identical to the authoritative outputs")

main_tex = (paths.MANUSCRIPT / "main.tex").read_text(encoding="utf-8")
si_tex = (paths.MANUSCRIPT / "SI.tex").read_text(encoding="utf-8")
all_tex = main_tex + si_tex

refs = _re.findall(r"\\(?:input|includegraphics(?:\[[^\]]*\])?)\{([^}]+)\}", all_tex)
missing = []
for ref in refs:
    p = (paths.MANUSCRIPT / ref).resolve()
    if not p.exists():
        missing.append(ref)
check(not missing, f"all {len(refs)} LaTeX \\input/\\includegraphics references resolve to existing files")

unreferenced = []
for _, r in trace.iterrows():
    fname = r["output_path"].split("/")[-1]
    if fname not in all_tex:
        unreferenced.append(r["artifact_id"])
check(not unreferenced, "every retained artifact is referenced by the main manuscript or the SI")

legacy_values = ["5.92", "3.65", "8.09", "7.73", "40.0\\%", "20.9\\%", "19.1 percentage", "19.1 pp"]
hits = [v for v in legacy_values if v in all_tex]
check(not hits, "no legacy primary dormant values remain in the manuscript or SI")
hardcoded = _re.findall(r"Tables?~?\s?S\d", all_tex)
check(not hardcoded, "no hard-coded SI table numbers remain (semantic references only)")

si_full = si_tex
for ref in _re.findall(r"\\input\{([^}]+)\}", si_tex):
    p = (paths.MANUSCRIPT / ref).resolve()
    if p.exists():
        si_full += p.read_text(encoding="utf-8")
si_labels = set(_re.findall(r"\\label\{((?:tab|fig):si-[^}]+)\}", si_full))
si_refs = set(_re.findall(r"\\ref\{((?:tab|fig):si-[^}]+)\}", si_full))
dangling = si_refs - si_labels
check(not dangling, "every semantic SI reference resolves to a defined label")
print(f"SI defines {len(si_labels)} semantic artifact labels")

# Hard visibility-support reconciliation: the Main text, the SI visibility-model
# table, Figure 3B's tercile support, and the SI selection table's realized row
# must all report exactly the continuous-model support from visibility.csv, and
# the notebook 01 exposure validation must reproduce it. Any unexplained
# inequality fails the run.
vis_v = R("visibility.csv").iloc[0]
vs_n, vs_songs = int(vis_v["n_observations"]), int(vis_v["n_clusters"])
sel_v = R("visibility_selection.csv")
sel_real_v = sel_v[sel_v["has_realized_boxoffice"]].iloc[0]
check(int(sel_real_v["n_song_snapshots"]) == vs_n and int(sel_real_v["n_unique_songs"]) == vs_songs,
      "visibility selection table realized row equals the continuous-model support")
check(int(R("visibility_terciles.csv")["n_song_snapshots"].sum()) == vs_n,
      "Figure 3B tercile support sums exactly to the continuous-model support")
val_v = R("time_respecting_exposure_validation.csv").set_index("quantity")["n"]
check(int(val_v["film-linked song-snapshots with realized box office"]) == vs_n
      and int(val_v["unique songs in the realized box-office support"]) == vs_songs,
      "notebook 01 exposure validation reproduces the continuous-model support")
check(f"$N={vs_n:,}$".replace(",", "{,}") in main_tex and f"{vs_songs:,} normalized song-artist identities" in main_tex,
      "Main text reports exactly the continuous visibility-model support")
check("1.15" in main_tex and "[0.95, 1.35]" in main_tex,
      "Main text and Figure 3 caption report the continuous visibility estimate and CI")
tex_s_models = (paths.OUT_TABLES_SI / "table_si_visibility_models.tex").read_text(encoding="utf-8")
tex_s_selection = (paths.OUT_TABLES_SI / "table_si_visibility_selection.tex").read_text(encoding="utf-8")
check(fmt_int(vs_n) in tex_s_models and fmt_int(vs_songs) in tex_s_models,
      "SI visibility-model table reports exactly the continuous-model support")
check(fmt_int(vs_n) in tex_s_selection and fmt_int(vs_songs) in tex_s_selection,
      "SI selection table renders exactly the continuous-model support")
print(f"visibility-support reconciliation PASS: {vs_n:,} obs / {vs_songs:,} songs consistent across Main, SI tables, Figure 3B, and the notebook 01 exposure validation")

# Typographic and content-integrity checks for the readability pass: no
# generated table may use \tiny or \resizebox; the priority tables must keep
# every information field; Main prose must be free of alignment artifacts; the
# Abstract, Significance Statement, and figure captions must respect length
# limits; and no orphan transition sentence may remain.
tiny_hits, resize_hits = [], []
for p_ in sorted(paths.OUT_TABLES_MAIN.glob("*.tex")) + sorted(paths.OUT_TABLES_SI.glob("*.tex")):
    t_ = p_.read_text(encoding="utf-8")
    if "\\tiny" in t_:
        tiny_hits.append(p_.name)
    if "\\resizebox" in t_:
        resize_hits.append(p_.name)
check(not tiny_hits, "no generated table uses tiny type")
check(not resize_hits, "no generated table uses resizebox scaling")

t2_tex = (paths.OUT_TABLES_MAIN / "table_memory_decay_key_results.tex").read_text(encoding="utf-8")
T2_ROWS = ["Pooled film-linked excess premium", "Film-embedded by snapshot", "Prior-attention matched excess premium",
           "Strict temporal event contrast", "Strict temporal post contrast", "Lower pre-film excess-attention contrast",
           "Exceeding the pre-film threshold"]
check(all(s in t2_tex for s in T2_ROWS), "Main Table 2 retains all seven data rows")
check(all(s in t2_tex for s in ["Result", "Estimate", "uncertainty", "decay-equivalent", "Interpretation"]),
      "Main Table 2 retains all five information fields")
check("\\begin{tabularx}" in t2_tex, "Main Table 2 uses an unscaled tabularx layout")

s8_tex = (paths.OUT_TABLES_SI / "table_si_matching_balance.tex").read_text(encoding="utf-8")
s21_tex = (paths.OUT_TABLES_SI / "table_si_catalog_balance.tex").read_text(encoding="utf-8")
for nm_, t_ in [("song balance", s8_tex), ("catalog balance", s21_tex)]:
    check("Before matching" in t_ and "After matching" in t_ and t_.count("T mean") == 4 and t_.count("C mean") == 4 and t_.count("SMD") >= 4,
          f"{nm_} table keeps the grouped two-level header with all treated/control mean and SMD columns")
    check("\\begin{longtable}" in t_ and "continued." in t_, f"{nm_} table is a longtable with continuation headers")

s23_tex = (paths.OUT_TABLES_SI / "table_si_catalog_retention.tex").read_text(encoding="utf-8")
check("Observed change pattern" in s23_tex, "catalog attention-change table keeps the observed-change-pattern column")
s25_tex = (paths.OUT_TABLES_SI / "table_si_cluster_inference.tex").read_text(encoding="utf-8")
check("Cluster-by-song inference" in s25_tex and "HC1" in s25_tex and "Support" in s25_tex,
      "cluster-inference table keeps grouped HC1 and cluster-by-song inference columns")
s26_tex = (paths.OUT_TABLES_SI / "table_si_robustness_summary.tex").read_text(encoding="utf-8")
check("Interpretation" in s26_tex and "Exact specification" in s26_tex,
      "robustness table keeps specification and interpretation fields")

check("&=" not in main_tex, "no malformed alignment character remains in Main prose")
check("This leaves the relational implication" not in main_tex, "no orphan transition sentence before the catalog section")

def _word_count(t_):
    t_ = _re.sub(r"\\[a-zA-Z]+\*?", " ", t_)
    return len([w for w in _re.sub(r"[{}$~\[\]]", " ", t_).split() if _re.search(r"[0-9A-Za-z]", w)])

abstract_body = _re.search(r"\\begin\{abstract\}(.*?)\\end\{abstract\}", main_tex, _re.S).group(1)
sig_body = _re.search(r"\\section\*\{Significance Statement\}(.*?)\\noindent\\textbf\{Keywords:", main_tex, _re.S).group(1)
check(_word_count(abstract_body) <= 250, f"Abstract is {_word_count(abstract_body)} words (limit 250)")
check(50 <= _word_count(sig_body) <= 120, f"Significance Statement is {_word_count(sig_body)} words (50-120)")

captions = [l_ for l_ in main_tex.splitlines() if l_.strip().startswith("\\caption{")]
check(len(captions) == 4, "Main has exactly four figure captions")
CAP_LIMIT = [160, 240, 195, 185]
for i_, (cap_, lim_) in enumerate(zip(captions, CAP_LIMIT), 1):
    wc_ = _word_count(cap_)
    check(wc_ <= lim_, f"Figure {i_} caption is {wc_} words (limit {lim_})")
# Micro-editorial closure checks: revised Abstract/Significance clauses, the
# association-based Discussion framing, distinct SI-section caption references,
# and the corrected Table S8 title.
check("18.0 percentage points higher" in main_tex, "Abstract reports the above-threshold probability difference")
check("relative to the fitted" in si_tex or "cross-sectional" in si_tex, "SI distinguishes the expected-attention baseline from individual decay")
check("observational account of cross-domain reactivation" in main_tex and "excess-attention level" in main_tex, "Main distinguishes attention levels from observational temporal reactivation")
check("can reactivate focal cultural objects" not in main_tex, "no direct mechanistic reactivation conclusion remains")
check("selectively reactivated by the present" not in main_tex, "the rhetorical archive sentence is removed")
check("Attention Regimes and Matched Song-Level Comparisons" not in main_tex
      and "Diagnostics and Dormant-Song Definitions" not in main_tex
      and "Film Visibility and Temporal Consistency Checks" not in main_tex,
      "no caption merges two SI section titles into one")
check(main_tex.count("SI Appendix section" + " " + chr(92) + "textit{") >= 2
      and (chr(92) + "textit{Temporal Consistency Checks}") in main_tex,
      "captions reference distinct SI sections in italics")
check("Covariate balance across song-level, strict-temporal, and prior-attention matched designs." in s8_tex,
      "Table S8 title covers all three matched families")
# SI ordering and pooled-premium consistency: the catalog-matching table must be
# pinned in source order ahead of the balance longtables, and the Main text must
# report the pooled Spotify premium rounded from the authoritative result object,
# matching the premiums and cluster-inference tables.
ap_v = R("attention_premiums.csv")
ppv = float(ap_v[ap_v["snapshot"].eq("pooled")].iloc[0]["estimate"])
check(f"{ppv:.2f}" == "13.79", "authoritative pooled Spotify premium rounds to 13.79")
check(f"The pooled Spotify premium across high-coverage snapshots is {ppv:.2f} points" in main_tex,
      "Main reports the pooled Spotify premium rounded from the authoritative result object")
check("13.80" not in main_tex and "13.80" not in si_full, "no stale 13.80 rounding remains anywhere")
s3_tex_v = (paths.OUT_TABLES_SI / "table_si_attention_premiums.tex").read_text(encoding="utf-8")
check(f"{ppv:.2f}" in s3_tex_v and f"{ppv:.2f}" in s25_tex,
      "the premiums table and the cluster-inference table report the same rounded pooled premium as Main")
s20_tex_v = (paths.OUT_TABLES_SI / "table_si_catalog_matching.tex").read_text(encoding="utf-8")
check("\\begin{table}[H]" in s20_tex_v, "catalog-matching table is pinned in source order before the balance longtables")
check("consolidates covariate-level balance across the song-level, strict-temporal, and prior-attention matched designs" in si_tex,
      "SI introduces the balance table with all three design families")
print("typographic and content-integrity checks PASS")

check("expected-memory baseline" not in main_tex.lower() + si_tex.lower(),
      "Main and SI consistently name the expected-attention baseline")
check("cross-sectional age--attention profiles" in main_tex and "longitudinal decay path" in si_tex,
      "Main and SI identify fitted age profiles as cross-sectional")
check("other Billboard songs" in main_tex and "complete repertoire" in si_tex,
      "catalog interpretation preserves the observed Billboard scope")
check("119 songs" in main_tex and "355 film-linked songs" in main_tex,
      "Main distinguishes the temporal subgroup from its eligible population")


PASS: manuscript Figures/Tables copies are byte-identical to the authoritative outputs
PASS: all 38 LaTeX \input/\includegraphics references resolve to existing files
PASS: every retained artifact is referenced by the main manuscript or the SI
PASS: no legacy primary dormant values remain in the manuscript or SI
PASS: no hard-coded SI table numbers remain (semantic references only)
PASS: every semantic SI reference resolves to a defined label
SI defines 32 semantic artifact labels
PASS: visibility selection table realized row equals the continuous-model support
PASS: Figure 3B tercile support sums exactly to the continuous-model support
PASS: notebook 01 exposure validation reproduces the continuous-model support
PASS: Main text reports exactly the continuous visibility-model support
PASS: Main text and Figure 3 caption report the continuous visibility estimate and CI
PASS: SI visibility-model table reports exactly the continuous-model support
PASS: SI selection table renders exactly t

## Compact artifact index by evidence family


In [10]:
index = trace.groupby("evidence_family").agg(
    artifacts=("artifact_id", lambda s: ", ".join(s)),
    n=("artifact_id", "size"),
).reset_index()
display(index)
print(f"Final artifact set: {len(trace)} retained numerical artifacts "
      f"({(trace['main_or_si'].eq('si')).sum()} SI numerical artifacts).")


,evidence_family,artifacts,n
0,artist-catalog attention,"figure4_artist_catalog_regimes, table_si_catal...",9
1,data construction and linkage,"data_audit_table, table_si_platform_coverage, ...",3
2,embedding and visibility,"table_si_repeated_embedding, table_si_visibili...",4
3,embedding and visibility; temporal consistency,figure3_embedding_visibility_temporal,1
4,expected-attention baselines,"table_memory_decay_key_results, table_si_memor...",5
5,expected-attention baselines; lower pre-film e...,figure2_expected_memory_excess_dormant,1
6,inference and robustness,"table_si_cluster_inference, table_si_robustnes...",2
7,lower pre-film excess-attention-song reactivation,"table_si_dormant_definitions, table_si_dormant...",4
8,matched song-level comparisons,"table_si_song_matching_estimates, table_si_son...",4
9,platform coverage and attention differences,"table_si_attention_premiums, figure_si_lastfm_...",2


Final artifact set: 38 retained numerical artifacts (32 SI numerical artifacts).
